In [ ]:
# 13_capacity_to_pay.ipynb
# Revision robustness: catastrophic health expenditure under CONSUMPTION-based
# capacity to pay (total consumption; WHO non-food/non-subsistence) rather than
# income, re-estimating the shock effect and the PHI buffer (TWFE and Callaway-
# Sant'Anna). Writes t14_* and fig16 (grayscale).
# Experiment C: alternative capacity-to-pay (consumption-based / WHO non-food) CHE,
# re-estimating the shock effect and the PHI buffer. Addresses income-vs-consumption
# denominator sensitivity (Cylus et al. 2018).
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, pyreadstat
from linearmodels.panel import PanelOLS
from differences import ATTgt
RAW="../data/raw"; PROC="../data/processed"; TAB="../results/tables"
WAVES=[(2019,"a"),(2020,"b"),(2021,"c"),(2022,"d"),(2023,"e"),(2024,"f")]
def num(s): return pd.to_numeric(s, errors="coerce")

# ---- household consumption from hh file: LIV_EXP (total), LIV_EXP1 food, LIV_EXP2 dining (monthly, 10,000 KRW) ----
hh=[]
for yr,pfx in WAVES:
    df,_=pyreadstat.read_sas7bdat(f"{RAW}/sas_{yr}/{pfx}_hh.sas7bdat", usecols=["HHID","LIV_EXP","LIV_EXP1","LIV_EXP2"])
    for c in ["LIV_EXP","LIV_EXP1","LIV_EXP2"]: df[c]=num(df[c]).where(lambda x:x>=0)
    df["year"]=yr; hh.append(df)
hh=pd.concat(hh, ignore_index=True)
hh["cons_tot"]=hh["LIV_EXP"]*12*10000.0                                  # annual total consumption (KRW)
hh["cons_food"]=(hh["LIV_EXP1"].fillna(0)+hh["LIV_EXP2"].fillna(0))*12*10000.0
hh["cons_nonfood"]=hh["cons_tot"]-hh["cons_food"]                        # WHO capacity-to-pay (net of subsistence food)
hh=hh.drop_duplicates(["HHID","year"])

p=pd.read_parquet(f"{PROC}/khp_panel.parquet")
p=p.merge(hh[["HHID","year","cons_tot","cons_nonfood"]], on=["HHID","year"], how="left")

for den,nm in [("cons_tot","totcons"),("cons_nonfood","nonfood")]:
    cap=p[den].where(lambda x:x>0)
    r=p["hh_oop"]/cap
    for th in [0.25,0.40]:
        p[f"che{int(th*100)}_{nm}"]=np.where(cap.notna(),(r>=th).astype(float),np.nan)

print("=== CHE incidence under alternative capacity-to-pay denominators ===")
print("income (baseline) : CHE25=%.4f CHE40=%.4f"%(p["che25"].dropna().mean(),p["che40"].dropna().mean()))
print("total consumption : CHE25=%.4f CHE40=%.4f"%(np.nanmean(p["che25_totcons"]),np.nanmean(p["che40_totcons"])))
print("non-food (WHO CTP): CHE25=%.4f CHE40=%.4f"%(np.nanmean(p["che25_nonfood"]),np.nanmean(p["che40_nonfood"])))
print("coverage (non-missing denominator): %.3f"%p["cons_tot"].notna().mean())

# ---- TWFE shock + buffer on the alternative outcomes ----
p=p.sort_values(["PIDWON","year"])
first=p[p["shock_any"]==1].groupby("PIDWON")["year"].min().rename("first_shock")
p=p.merge(first,on="PIDWON",how="left")
p["ever"]=p["first_shock"].notna().astype(int)
p["post_shock"]=((p["year"]>=p["first_shock"])&(p["ever"]==1)).astype(int)
p["post_phi"]=p["post_shock"]*p["base_phi"]
cnt=p.groupby("PIDWON")["year"].transform("size"); est=p[cnt>=2].set_index(["PIDWON","year"])
def fe(y,X):
    d=est[[y]+X].dropna(); return PanelOLS(d[y],d[X],entity_effects=True,time_effects=True).fit(cov_type="clustered",cluster_entity=True)
print("\n=== TWFE shock effect and PHI buffer under alternative CHE ===")
rows=[]
for y,lab in [("che40_totcons","CHE40 total-consumption"),("che25_totcons","CHE25 total-consumption"),
              ("che40_nonfood","CHE40 non-food (WHO)"),("che25_nonfood","CHE25 non-food (WHO)")]:
    r1=fe(y,["post_shock"]); r2=fe(y,["post_shock","post_phi"])
    rows.append({"outcome":lab,"shock":r1.params["post_shock"],"shock_se":r1.std_errors["post_shock"],
                 "buffer":r2.params["post_phi"],"buffer_se":r2.std_errors["post_phi"],"buffer_p":r2.pvalues["post_phi"]})
alt=pd.DataFrame(rows); alt.to_csv(f"{TAB}/t14_che_capacity_to_pay.csv", index=False)
print(alt.round(4).to_string(index=False))

# ---- CS overall + buffer on non-food CHE40 (primary robustness) ----
def cs(df,outcome):
    d=df[["PIDWON","year","cohort",outcome]].dropna(subset=[outcome]).copy()
    d["cohort"]=d["cohort"].replace(0,np.nan); d=d.set_index(["PIDWON","year"])
    att=ATTgt(data=d,cohort_column="cohort",base_period="universal")
    att.fit(formula=outcome,est_method="dr",control_group="not_yet_treated",progress_bar=False)
    s=att.aggregate("simple"); s.columns=[str(c) for c in s.columns]; return float(s.iloc[0,0]),float(s.iloc[0,1])
sa,sea=cs(p,"che40_nonfood"); s0,se0=cs(p[p["base_phi"]==0],"che40_nonfood"); s1,se1=cs(p[p["base_phi"]==1],"che40_nonfood")
print("\n=== CS (non-food CHE40): overall=%.4f(se %.4f) | buffer No-PHI=%.4f(se %.4f) Has-PHI=%.4f(se %.4f) ==="%(sa,sea,s0,se0,s1,se1))
pd.DataFrame([{"outcome":"che40_nonfood","overall":sa,"overall_se":sea,"noPHI":s0,"noPHI_se":se0,"hasPHI":s1,"hasPHI_se":se1}]).to_csv(f"{TAB}/t14_che_nonfood_cs.csv",index=False)


# ---- Figure 16: buffer under three capacity-to-pay denominators ----
import seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size":11,"axes.edgecolor":"0.3","savefig.dpi":600,"figure.dpi":110,"grid.color":"0.9"})
inc=pd.read_csv(f"{TAB}/t03_static_fe.csv"); alt=pd.read_csv(f"{TAB}/t14_che_capacity_to_pay.csv")
inc_b=float(inc[inc["outcome"]=="CHE 40%"]["buffer_postPHI"].iloc[0]); inc_se=float(inc[inc["outcome"]=="CHE 40%"]["buffer_se"].iloc[0])
tc=alt[alt["outcome"]=="CHE40 total-consumption"].iloc[0]; nf=alt[alt["outcome"]=="CHE40 non-food (WHO)"].iloc[0]
labels=["Income\n(baseline)","Total\nconsumption","Non-food\n(WHO CTP)"]
vals=[inc_b*100,tc["buffer"]*100,nf["buffer"]*100]; ses=[inc_se*100,tc["buffer_se"]*100,nf["buffer_se"]*100]
fig,ax=plt.subplots(figsize=(6.4,4.3))
b=ax.bar(labels,vals,yerr=[1.96*s for s in ses],color=["0.25","0.5","0.7"],edgecolor="0.3",capsize=5,width=0.6)
ax.axhline(0,color="0.5",lw=0.8)
for bi,v in zip(b,vals): ax.text(bi.get_x()+bi.get_width()/2,v-0.35,f"{v:.1f}pp",ha="center",va="top",fontweight="bold",color="white")
ax.set_ylabel("PHI buffer on P(CHE 40%): shock x baseline-PHI, pp"); ax.set_xlabel("Capacity-to-pay denominator")
for ext in ("png","pdf"): fig.savefig(f"../results/figures/fig16_capacity_to_pay.{ext}",dpi=600,bbox_inches="tight")
plt.close(fig); print("fig16 written")
